# Reranking for Better Retrieval

Embedding similarity is fast but approximate — it scores a query against every chunk independently, without ever looking at the two together. A **reranker** fixes this as a second pass: retrieve a wider net of candidates cheaply with embeddings, then re-score just those candidates with a slower but more accurate cross-encoder model that reads the query and each chunk together.

This runs entirely locally — no API key, no external service — using a small cross-encoder from the `sentence-transformers` library.


**Step 1 — Set up (and quiet down Hugging Face).** Configures the usual LLM/embedding models, plus a few environment variables that silence the local cross-encoder model's download progress bars and warnings so the output below stays clean.


In [1]:
import logging
import os
import warnings

# The cross-encoder reranker below downloads a model from Hugging Face — these
# three lines silence its progress bars, verbose logging, and tokenizer
# warnings so the printed output stays focused on our own results.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet down noisy INFO-level logs from the HTTP client, LlamaIndex, and the
# sentence-transformers / huggingface_hub libraries the reranker depends on.
for noisy_logger in ("httpx", "llama_index", "sentence_transformers", "huggingface_hub"):
    logging.getLogger(noisy_logger).setLevel(logging.ERROR)

# Loads .env into os.environ so OPENAI_API_KEY is available to the clients below.
load_dotenv()

Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 2 — Retrieve a wide net with embeddings.** `similarity_top_k=5` pulls back the 5 nodes with the highest cosine similarity to the question — fast, but only an approximation of true relevance since query and chunk are scored independently.


In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)

question = "What is the protagonist's strongest ability or transformation?"
retriever = index.as_retriever(similarity_top_k=5)  # cast a wide net: top 5 nodes by embedding similarity
retrieved_nodes = retriever.retrieve(question)

print("--- Initial retrieval order (embedding similarity) ---")
for node in retrieved_nodes:
    print(f"{node.score:.3f} | {node.node.get_content()[:80]}")

--- Initial retrieval order (embedding similarity) ---
0.329 | Dragon Ball — Series Overview

Overview
Dragon Ball is a Japanese manga series c
0.313 | Demon Slayer — Series Overview

Overview
Demon Slayer (Kimetsu no Yaiba) is a Ja
0.300 | Solo Leveling — Series Overview

Overview
Solo Leveling originated as a web nove
0.243 | Trivia
Demon Slayer: Mugen Train became the highest-grossing Japanese film of al
0.229 | Naruto — Series Overview

Overview
Naruto is a Japanese manga series written and


**Step 3 — Rerank with a cross-encoder.** `SentenceTransformerRerank` re-scores each of those 5 candidates by feeding the question and chunk into the model together, then keeps only the top 2 — slower per-item, but far more accurate than embedding similarity alone.


In [3]:
from llama_index.core.postprocessor import SentenceTransformerRerank

# Loads a small local cross-encoder model that scores (query, chunk) pairs
# jointly, rather than comparing two separately-computed embedding vectors.
reranker = SentenceTransformerRerank(top_n=2, model="cross-encoder/ms-marco-MiniLM-L-2-v2")
reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=question)  # re-score and keep only the top 2

print("--- After reranking (cross-encoder relevance score) ---")
for node in reranked_nodes:
    print(f"{node.score:.3f} | {node.node.get_content()[:80]}")
    print("---")

2026-09-08 09:53:35,930 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


--- After reranking (cross-encoder relevance score) ---
-3.900 | Solo Leveling — Series Overview

Overview
Solo Leveling originated as a web nove
---
-4.432 | Naruto — Series Overview

Overview
Naruto is a Japanese manga series written and
---


**Step 4 — Wire the reranker into a query engine.** Passing `node_postprocessors=[reranker]` makes every `.query()` call automatically retrieve wide, then rerank down to the best nodes before the LLM ever sees them — no need to call the retriever and reranker separately anymore.


In [4]:
# node_postprocessors run after retrieval but before the LLM sees the nodes —
# passing the reranker here applies the same retrieve-then-rerank flow
# automatically on every .query() call.
reranking_engine = index.as_query_engine(
    similarity_top_k=5,
    node_postprocessors=[reranker],
)
response = reranking_engine.query(question)
print(response)

The protagonist's strongest ability is his power as the Shadow Monarch, which allows him to extract defeated enemies as loyal shadow soldiers he can summon to fight for him.


### Summary

- Retrieval and reranking use two different, non-comparable scoring systems — embedding cosine similarity for the initial wide net, then cross-encoder relevance for the precise final ordering.
- The two-stage pattern (retrieve wide with `similarity_top_k`, then narrow with a `node_postprocessors` reranker) is standard because cross-encoders are far too slow to run against an entire corpus directly — they only make sense over a small pre-filtered candidate set.
